### 1.2 Load Your API Key Securely

We **never** hardcode API keys. Instead we keep them in a `.env` file and load them at runtime with `python-dotenv`. Treat your key like a password — if it leaks, anyone can run up charges on your account.


In [1]:
import os
from dotenv import load_dotenv

import textwrap


#This is optional. I use VPN in my computer. Why I need this. 
import truststore
truststore.inject_into_ssl()



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

load_dotenv('/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/openai_key.env')  # reads .env file in the current directory

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")

API key loaded successfully.


In [2]:
from openai import OpenAI

client = OpenAI(api_key=api_key)
pretty_print("OpenAI client ready.")

OpenAI client ready.


link to Documentation

[Chat Completions](https://developers.openai.com/api/reference/python/resources/chat/subresources/completions/methods/create)

[Responses API](https://developers.openai.com/api/reference/python/resources/responses/methods/create)

# Let's now go through responses API

| Feature             | Chat Completions API                             | Responses API                                              |
| ------------------- | ------------------------------------------------ | ---------------------------------------------------------- |
| **Endpoint**        | `client.chat.completions.create()`               | `client.responses.create()`                                |
| **Input format**    | `messages=[{"role": ..., "content": ...}]`       | `input=` (string or list of message dicts)                 |
| **System prompt**   | `{"role": "system", "content": ...}` in messages | `instructions=` parameter (top-level)                      |
| **Output access**   | `resp.choices[0].message.content`                | `resp.output_text`                                         |
| **Multi-turn**      | Manually pass full message history each time     | `previous_response_id=resp.id` (server-side context)       |
| **Developer role**  | Not supported (use `system`)                     | `{"role": "developer"}` for meta-instructions              |
| **Vision input**    | `{"type": "image_url", "image_url": {...}}`      | `{"type": "input_image", "image_url": ...}`                |
| **Reasoning / CoT** | Not natively supported                           | `reasoning={"effort": ..., "summary": ...}` built-in       |
| **Response object** | `ChatCompletion` with `choices[]` list           | `Response` with `output[]` list and `output_text` shortcut |
| **Streaming**       | `stream=True` yields `ChatCompletionChunk`       | `stream=True` yields server-sent events                    |
| **Tool calls**      | Supported via `tools` param                      | Supported via `tools` param (same)                         |
| **Model support**   | All chat models                                  | All chat models (newer, recommended going forward)         |


## Diff b/w Chat Completions and Responses API - Structure

In [3]:

# do the same with chat completions
resp = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
		{"role": "system", "content": "You are a friendly Python tutor."},
        {"role": "user", "content": "What is a list comprehension?"}
    ]
)
pretty_print("Chat Completions output:", resp.choices[0].message.content)


Chat Completions output: A list comprehension is a concise way to create a new
list by transforming elements from an existing iterable (like a list or range),
and optionally filtering which elements to include.  Basic form - [expression
for item in iterable]  Example: make a list of squares - squares = [x*x for x in
range(10)]  Adding a filter - [expression for item in iterable if condition]
Example: only even numbers - evens = [n for n in range(20) if n % 2 == 0]  Other
handy forms - Multiple sources (nested loops)   - pairs = [(i, j) for i in
range(3) for j in range(2)] - If-else inside the expression   - kind = [ 'even'
if n % 2 == 0 else 'odd' for n in range(5) ]  Equivalent (for learning) - A for-
loop version that builds the same list:   - squares = []   - for x in range(10):
-     squares.append(x*x)  Benefits - More concise and readable for simple
transformations. - In many cases faster than building a list with append.  Notes
- List comprehensions create a new list in memory. 

In [4]:

resp = client.responses.create(
    model="gpt-5-nano",
    instructions="You are a friendly Python tutor.",
    input="What is a list comprehension?"
)
pretty_print("Responses API output:", resp.output_text)




Responses API output: A list comprehension is a compact way to create a new list
by applying an expression to each item in an existing iterable, possibly
filtering some items, in a single line.  Syntax - [expression for item in
iterable [if condition]]  - The expression can be any computation using the
item. - The optional if filters elements.  Examples - Squares of numbers 0
through 9:   [x*x for x in range(10)]  - Even squares only:   [x*x for x in
range(10) if x % 2 == 0]  - Flatten a 2D list (matrix):   [elem for row in
matrix for elem in row]  - Conditional expression inside the result:   [x if x %
2 == 0 else -x for x in range(6)]   (gives [0, -1, 2, -3, 4, -5])  Notes - A
list comprehension creates a list in memory right away. - It’s often more
readable and concise than a for-loop with append, but avoid making it too long
or hard to read.  Related ideas - Generator expression (lazy, uses parentheses):
(x*x for x in range(10)) - Set/dict comprehensions:   - {expr for item in
iter

## Passing Multi Turn Conversation History to Responses API

In [5]:
input_messages = [
    {"role": "user", "content": "Why is Trump a jerk?"},
    {"role": "assistant", "content": "Some people are born that way."},
    {"role": "user", "content": "Name a celebrity who is not a jerk."}
]


resp = client.responses.create(
    model="gpt-5-nano",
    instructions="You are a very candid Journalist.",
    input=input_messages
)
pretty_print("Responses API output:", resp.output_text)

Responses API output: Public perception varies, but here are a few celebrities
who are widely regarded as not jerks:  - Keanu Reeves — frequently praised for
humility, politeness, and generosity; lots of fan anecdotes about his down-to-
earth nature. - Tom Hanks — often described as warm, gracious with fans, and
professional. - Dwayne Johnson — known for positivity, encouragement, and
philanthropic work.  Want me to pull up specific stories or receipts behind
these reputations?


In [6]:
input_messages = [
    {"role": "user", "content": "Why is Trump a jerk?"},
    {"role": "assistant", "content": "Some people are born that way."},
    {"role": "user", "content": "Name a celebrity who is not a jerk."}
]


resp = client.responses.create(
    model="gpt-5-nano",
    instructions="You are a very candid Journalist.",
    input=input_messages, 
	max_output_tokens=200,  # in responses there's max_output_tokens instead of max_tokens
	# temperature is NOT supported by reasoning models like gpt-5-nano
	reasoning={"effort": "minimal"},   
)
pretty_print("Responses API output:", resp.output_text)

Responses API output: That’s a tricky claim to prove, since “jerk” is
subjective. If you want a celebrity known for generally positive behavior and
how they treat people, you might point to someone like Keanu Reeves. Widely
admired for humility, kindness in interviews, and acts of generosity, many
people treat him as a contrast to the usual messy celebrity stereotype. Of
course, public personas are curated, but Reeves tends to get cited as non-jerky
by fans and outlets.


In [7]:
resp = client.responses.create(
    model="gpt-5-nano",
    instructions="You are a very candid journalist. Reply in 1 short sentence, starting with name of celeb.",
    input=input_messages,
    max_output_tokens=1000,
    reasoning={"effort": "high"},   # or "none"
    text={"verbosity": "low"},
)
print(repr(resp.output_text), resp.status, resp.incomplete_details)

'' incomplete IncompleteDetails(reason='max_output_tokens')


In [8]:
resp

Response(id='resp_0525f3fd44c0a273006a11268b64f0819e844934aae0950a3c', created_at=1779508875.0, error=None, incomplete_details=IncompleteDetails(reason='max_output_tokens'), instructions='You are a very candid journalist. Reply in 1 short sentence, starting with name of celeb.', metadata={}, model='gpt-5-nano-2025-08-07', object='response', output=[ResponseReasoningItem(id='rs_0525f3fd44c0a273006a11268c4960819eb18a4fc0c04fca03', summary=[], type='reasoning', content=None, encrypted_content=None, status=None)], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, completed_at=None, conversation=None, max_output_tokens=1000, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention='in_memory', reasoning=Reasoning(effort='high', generate_summary=None, summary=None, context='current_turn'), safety_identifier=None, service_tier='default', status='incomplete', text=ResponseTextConfig(format=Res

## Key Differences: Parameters in Responses API vs Chat Completions API

### `max_tokens` → `max_output_tokens`
In the **Responses API**, the parameter to limit output length is **`max_output_tokens`**, NOT `max_tokens`.

```python
# Chat Completions API
client.chat.completions.create(model="gpt-5-nano", messages=..., max_tokens=50)

# Responses API
client.responses.create(model="gpt-5-nano", input=..., max_output_tokens=50)
```

---

### `temperature` — Not Supported on Reasoning Models (GPT-5 family)

The entire GPT-5 family (`gpt-5`, `gpt-5-mini`, `gpt-5-nano`) are **reasoning models**. They do **NOT** support `temperature` or `top_p`.

| Model Family | Type | `temperature` | `top_p` | `max_output_tokens` |
|---|---|---|---|---|
| **gpt-4o / gpt-5-nano** | Non-reasoning | ✅ Supported | ✅ Supported | ✅ Supported |
| **gpt-5 / gpt-5-mini / gpt-5-nano** | Reasoning | ❌ Not supported | ❌ Not supported | ✅ Supported |

---

### How to Influence Creativity in GPT-5 Reasoning Models

Since `temperature` is locked, you control creativity through:

**1. `reasoning.effort` parameter** — controls how deeply the model thinks:
- `"low"` → concise, more deterministic
- `"medium"` → balanced
- `"high"` → deeper reasoning, more elaborate and exploratory

```python
resp = client.responses.create(
    model="gpt-5-nano",
    reasoning={"effort": "high", "summary": "auto"},
    input="Write a creative poem about Python."
)
```

**2. Prompt Engineering** — steer creativity through instructions:
```python
resp = client.responses.create(
    model="gpt-5-nano",
    instructions="Be wildly creative. Use unexpected metaphors.",
    input="Write a poem about Python."
)
```

**Bottom line:** With GPT-5 models, creativity = `reasoning.effort` + prompt wording, not `temperature`.

# Refer to previous responses

## Store True

In [9]:
# Turn 1
message1 = 'What is a list comprehension?'


resp1 = client.responses.create(
    model="gpt-5-nano",
    instructions="You are a friendly Python tutor.",
    input=message1,
	reasoning={"effort": "minimal"},   
    text={"verbosity": "low"}
)
pretty_print("Turn 1:", resp1.output_text)


Turn 1: A list comprehension is a concise way to create a new list by applying
an expression to each item in an iterable (and optionally filtering items). It
combines loop, condition, and expression in one readable line.  Syntax: - Basic:
[expression for item in iterable] - With filter: [expression for item in
iterable if condition]  Examples: - Squares of numbers 0–4: [x*x for x in
range(5)]   -> [0, 1, 4, 9, 16] - Even numbers from 0 to 9: [n for n in
range(10) if n % 2 == 0]  Benefits: shorter code, often faster, easy to read
once you’re familiar.


In [10]:

# Turn 2 — just pass previous_response_id, no history needed!
resp2 = client.responses.create(
    model="gpt-5-nano",
    input="Can you give me an example?",
    previous_response_id=resp1.id,  # <-- this is the magic
	reasoning={"effort": "minimal"},   
    text={"verbosity": "low"}
)
pretty_print("Turn 2:", resp2.output_text)


print()
print()
#or 


input_messages = [
    {"role": "user", "content": "What is a list comprehension?"},
    {"role": "assistant", "content": resp1.output_text},
    {"role": "user", "content": "Can you give me an example?"}
]

resp2_1 = client.responses.create(
    model="gpt-5-nano",
    instructions="You are a friendly Python tutor.",
    input=input_messages,
    reasoning={"effort": "minimal"},
    text={"verbosity": "low"}
)
pretty_print("Turn 2_1:", resp2_1.output_text)

Turn 2: Sure. Here’s a simple example in Python:  Goal: create a list of squares
for numbers 0 through 9.  Code: squares = [x*x for x in range(10)]
print(squares)  Output: [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]


Turn 2_1: Sure. Example: create a list of uppercase names from a list of mixed-
case names, but only include those longer than 3 letters.  names = ["alice",
"BOB", "Chad", "-eve", "Jo"] upper_long = [name.upper() for name in names if
len(name) > 3]  print(upper_long)  # ['ALICE', 'CHAD']


Why GPT-5 Models Give Different Outputs Each Time

- **No temperature control** — you can't set it to 0
- **Reasoning process is inherently non-deterministic** — the internal chain-of-thought exploration can branch differently each run, even with the same input
- **`reasoning.effort` is NOT the same as `temperature`** — "minimal" means "think less", not "be deterministic".


if you truly need deterministic outputs, use a non-reasoning model like gpt-4o-mini with temperature=0

In [11]:

# Turn 3 — chains from turn 2 (which already includes turn 1)
resp3 = client.responses.create(
    model="gpt-5-nano",
    input="What was my first question?",
    previous_response_id=resp2.id,
	reasoning={"effort": "minimal"},   
    text={"verbosity": "low"}
)
pretty_print("Turn 3:", resp3.output_text)

Turn 3: Your first question was: "What is a list comprehension?"


## Store False

In [12]:
# Turn 1
resp4 = client.responses.create(
    model="gpt-5-nano",
    instructions="You are a friendly Python tutor.",
    input="What is a list comprehension?",
	reasoning={"effort": "minimal"},   
    text={"verbosity": "low"},
	store=False
)
pretty_print("Turn 4:", resp4.output_text)

Turn 4: A list comprehension is a compact way to create a list in Python. It
combines looping, optional conditionals, and an expression into a single
readable line.  Syntax: - Basic: [expression for item in iterable] - With
condition: [expression for item in iterable if condition] - With transformation:
[expression for item in iterable if condition else other]  Examples: - Squares
0–9: [x*x for x in range(10)] - Even numbers from 0–19: [x for x in range(20) if
x % 2 == 0] - Convert names to lengths: [len(name) for name in ["Ana", "Bob",
"Cathy"]]  Benefits: - More concise than a loop - Often faster due to Python
internals - Easy to read once familiar with the pattern  Limitations: - Can hurt
readability if overly complex; prefer simple cases.


In [13]:

# Turn 5 — chains from turn 4 

try:
    resp5 = client.responses.create(
        model="gpt-5-nano",
        input="What was my first question?",
        previous_response_id=resp4.id,
        reasoning={"effort": "minimal"},   
        text={"verbosity": "low"}
    )
    pretty_print("Turn 5:", resp5.output_text)
except Exception as e:
    pretty_print("Error creating response:", str(e))

Error creating response: Error code: 400 - {'error': {'message': "Previous
response with id 'resp_019bbea739f32d90016a11269d118c81a2a2438c49d29d376f' not
found.", 'type': 'invalid_request_error', 'param': 'previous_response_id',
'code': 'previous_response_not_found'}}


# Nature of Instruction Prompt

## Scenario 1

In [14]:

# Turn 1: internal triage note that your backend stores in Zendesk as JSON
r1 = client.responses.create(
    model="gpt-5-nano",
    instructions="You are an internal support triage bot. Return only valid JSON with keys severity,suspected_causes,next_questions and do not write any customer-facing text.",
    input="A customer reports webhook deliveries started retrying heavily since 10:42 UTC and they see 502 errors from our endpoint on the Pro plan."
)

print("TURN 1:\n", r1.output_text)


TURN 1:
 {
  "severity": "Sev-1",
  "suspected_causes": [
    "Gateway/proxy layer returning 502 due to upstream service health/dependency outage",
    "Recent deployment or configuration change in the webhook delivery service causing upstream failures",
    "Destination endpoints (customer webhooks) are returning 5xx errors, causing our gateway to respond 502",
    "Network/DNS connectivity issues between our platform and destination endpoints or within our own infrastructure",
    "Traffic spike or queue backpressure leading to overwhelmed workers and 502 responses",
    "TLS/Certificate issues or handshake failures at the destination causing upstream failures",
    "Regional outage or degraded capacity affecting the webhook delivery path"
  ],
  "next_questions": [
    "Are the 502 errors affecting all destinations or only a subset (please provide example destination URLs and counts)?",
    "Did you recently change any destination configurations (URL changes, TLS certificates, DNS r

In [15]:

# Turn 2: continue the chain, but DON'T pass instructions again
# Ask for customer-facing email (this conflicts with Turn 1 rules)
r2 = client.responses.create(
    model="gpt-5-nano",
    previous_response_id=r1.id,
    input="Now write a customer-facing email reply: apologize, explain what we’re checking, and ask for 2 specific details. Plain English, not JSON."
)

print("\nTURN 2:\n", r2.output_text)


TURN 2:
 Subject: We’re investigating the webhook 502 errors on your Pro plan

Hi there,

I’m sorry for the disruption you’re seeing. Webhook deliveries have been retrying heavily since 10:42 UTC, and you’re getting 502 errors from our endpoint. We’re treating this as a high-priority issue and are actively investigating.

What we’re checking now:
- Our delivery gateway and any recent deployments or config changes that could cause upstream failures.
- Whether destination endpoints are returning errors or are temporarily unreachable.
- Network/DNS connectivity between our systems and your endpoints, plus any regional impacts.
- Possible backpressure or export volume spikes and TLS/certificate conditions that could affect handshakes.

Two details would help us diagnose this faster:
- Please share a recent webhook delivery sample, including: delivery_id, timestamp (UTC), destination URL, the observed 502 status, and any response body or headers you saw.
- Do these failures affect all dest

## Scenario 2

In [16]:


r1 = client.responses.create(
    model="gpt-5-nano",
    store=True,
    instructions="Output only a Markdown table with columns Category,Score(1-5),Evidence and no text outside the table.",
    input="Interview notes say the candidate built an end-to-end RAG demo on Azure, has strong system design, is weaker on fundamentals like precision/recall, communicates clearly, and gets defensive on feedback."
)

print("TURN 1:\n", r1.output_text)


TURN 1:
 | Category | Score(1-5) | Evidence |
| --- | --- | --- |
| End-to-end RAG Demo on Azure | 4 | Built an end-to-end RAG demo on Azure. |
| System Design | 5 | Has strong system design. |
| Fundamentals (Precision/Recall) | 2 | Weaker on fundamentals like precision/recall. |
| Communication | 4 | Communicates clearly. |
| Coachability / Feedback Receptiveness | 2 | Gets defensive on feedback. |


In [17]:

r2 = client.responses.create(
    model="gpt-5-nano",
    previous_response_id=r1.id,
    input="Now draft a polite rejection email in 6-8 sentences with a warm tone and do not include any table."
)

print("\nTURN 2:\n", r2.output_text)


TURN 2:
 Dear [Candidate Name],

Thank you for taking the time to interview with us for the [Role] position and for sharing your work on the Azure RAG demo. We were impressed by your end-to-end approach and your strong system design. At this time, we have decided to move forward with another candidate whose background more closely aligns with the current needs. Your clear communication and thoughtful approach were evident, and we appreciate the insights you brought to the conversation. One area we would encourage continued growth in is precision/recall fundamentals and how to receive feedback constructively in real time. We believe you have a lot to contribute in future roles and would welcome you to apply again. Thank you again for your time and all the effort you put into the process.

Warm regards,

[Your Name]
[Your Title]
[Company]


# Developer Role and Meta Instructions

## Scenario 1

In [18]:
r1 = client.responses.create(
    model="gpt-5-nano",
    store=True,
    input=[
        {"role": "developer", "content": "You are an internal support triage assistant and you must always output only valid JSON with keys severity,suspected_causes,next_questions and never produce customer-facing prose."},
        {"role": "user", "content": "Customer reports webhook deliveries started retrying heavily since 10:42 UTC and they see 502 errors from our endpoint on the Pro plan."}
    ],
)

print("TURN 1:\n", r1.output_text)


TURN 1:
 {
  "severity": "critical",
  "suspected_causes": [
    "Customer endpoint(s) behind a backend returning 502 Bad Gateway due to backend instability, crash, or misconfiguration",
    "Issue in our webhook delivery path (gateway/proxy or upstream service) returning 502 due to unhealthy backends or recent config changes",
    "Network connectivity problems between our system and the customer's endpoint (DNS, firewall, IP allowlists, intermittent routing)",
    "SSL/TLS handshake or certificate problems on the customer's endpoint causing upstream failures",
    "Recent changes to webhook configuration (endpoint URL, authentication headers) or rate-limiting on the customer's side"
  ],
  "next_questions": [
    "What is the exact endpoint URL(s) being delivered to, and are 502s observed for all webhooks or only a subset?",
    "Are 502 errors seen across all regions/environments or only in specific ones?",
    "When did the issue start (confirm if 10:42 UTC is the first observed ti

In [19]:

r2 = client.responses.create(
    model="gpt-5-nano",
    previous_response_id=r1.id,
    input="Now write a customer-facing email apology explaining what we’re checking and ask for exactly two specific details in plain English.",
    reasoning={"effort": "minimal"},   
    text={"verbosity": "low"}
)

print("\nTURN 2:\n", r2.output_text)


TURN 2:
 {
  "severity": "critical",
  "suspected_causes": [
    "Backend instability or misconfiguration causing 502 responses on the webhook delivery path",
    "Temporary issues in our webhook gateway or upstream services",
    "Network or TLS-related problems affecting delivery to the customer endpoint",
    "Recent changes to webhook configuration or rate limits impacting delivery"
  ],
  "next_questions": [
    "Please provide the exact endpoint URL that is experiencing the failures.",
    "Please share the precise time (in UTC) when you first noticed the failures and whether it has persisted since then."
  ]
}


## Scenario 2

In [20]:
r1 = client.responses.create(
    model="gpt-5-nano",
    store=True,
    input=[
        {"role": "developer", "content": "You are an interviewer note assistant and you must always output only a Markdown table with columns Category,Score(1-5),Evidence and no text outside the table."},
        {"role": "user", "content": "Interview notes: candidate built an end-to-end RAG demo on Azure, strong system design, confused precision vs recall, communicates clearly, slightly defensive on feedback."}
    ],
)
print("TURN 1:\n", r1.output_text)

TURN 1:
 | Category | Score(1-5) | Evidence |
|---|---|---|
| End-to-End Solution | 5 | Built an end-to-end RAG demo on Azure |
| Azure/Cloud Expertise | 5 | Demonstrated working implementation on Azure, indicating strong cloud skills |
| System Design | 5 | Noted for strong system design in the interview notes |
| ML Concepts (Precision vs Recall) | 2 | Confused understanding of precision vs recall |
| Communication | 5 | Communicates clearly |
| Feedback Receptiveness | 2 | Slightly defensive on feedback |


In [21]:



r2 = client.responses.create(
    model="gpt-5-nano",
    previous_response_id=r1.id,
    input="Now draft a polite rejection email in 6-8 sentences with a warm tone.",
)

print("\nTURN 2:\n", r2.output_text)


TURN 2:
 | Category | Score(1-5) | Evidence |
|---|---|---|
| Rejection Email (Polite, Warm) | 5 | Dear [Candidate Name], Thank you for taking the time to speak with us about the [Role] position. We enjoyed learning about your experience with RAG and Azure, and your thoughtful approach to end-to-end solutions. After careful consideration, we have decided to move forward with other candidates whose experience more closely aligns with our current needs. This was a difficult decision given your strengths in system design and your clear communication. We will keep your resume on file and will reach out if future opportunities align with your background. We genuinely appreciate your interest in our team and wish you success in your ongoing job search. If you would like feedback on your interview, we would be happy to share at your convenience. Thank you again for your time and effort. |


In [22]:



r2_1 = client.responses.create(
    model="gpt-5-nano",
    previous_response_id=r1.id,
    input="Now draft a polite rejection email in 6-8 sentences with a warm tone and I want reply in simple text, no fancy tables or nothing, got it? . ",
)

print("\nTURN 2:\n", r2_1.output_text)


TURN 2:
 | Category | Score(1-5) | Evidence |
|---|---|---|
| Overall Fit for Role | 3 | Mixed strengths (Azure, end-to-end RAG demo, and system design) but gaps in ML concepts and defensive feedback style could hinder collaboration. |
| End-to-End Solution | 5 | Built an end-to-end RAG demo on Azure. |
| Azure/Cloud Expertise | 5 | Demonstrated working implementation on Azure. |
| System Design | 5 | Strong system design. |
| ML Concepts (Precision vs Recall) | 2 | Demonstrates confusion between precision and recall. |
| Communication | 5 | Communicates clearly. |
| Feedback Receptiveness | 2 | Slightly defensive on feedback. |
| Collaboration/Team Fit | 3 | Defensive feedback stance may affect openness to input. |


In [23]:
r2_1 

Response(id='resp_039332a3ae889dbf006a11270582c08196aae0542ca6559c0a', created_at=1779508997.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5-nano-2025-08-07', object='response', output=[ResponseReasoningItem(id='rs_039332a3ae889dbf006a112705c12881968d41666b80968235', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseOutputMessage(id='msg_039332a3ae889dbf006a11270cb12481969dbb3776b3aab85d', content=[ResponseOutputText(annotations=[], text='| Category | Score(1-5) | Evidence |\n|---|---|---|\n| Overall Fit for Role | 3 | Mixed strengths (Azure, end-to-end RAG demo, and system design) but gaps in ML concepts and defensive feedback style could hinder collaboration. |\n| End-to-End Solution | 5 | Built an end-to-end RAG demo on Azure. |\n| Azure/Cloud Expertise | 5 | Demonstrated working implementation on Azure. |\n| System Design | 5 | Strong system design. |\n| ML Concepts (Precision vs Recall) | 2 | Demonstrates

# Let's understand with examples how  instructions and developer prompt combine in real world use case.

## Scenario 1: Helpdesk ticket pipeline (route the ticket → reply to customer)

You want the assistant to always obey company policy (developer), but you sometimes need strict JSON for automation and sometimes a human email for the customer. If you put “JSON-only” in developer, you’d break the email step; if you put policy in instructions, you must resend it every call.

In [24]:
r1 = client.responses.create(
    model="gpt-5-nano",
    input=[
        {"role": "developer", "content": "You are ACME Support; never invent account-specific facts; if info is missing ask at most two clarifying questions; do not reveal internal policies or tools."},
        {"role": "user", "content": "Customer: Since 10:42 UTC our webhooks keep retrying and we see lots of 502s; started after we enabled v2 signing; impact is EU customers."},
    ],
    instructions="Return only valid JSON with keys queue, severity, suspected_component, next_questions.",
)

print(r1.output_text)

{
  "queue": "webhook_delivery",
  "severity": "critical",
  "suspected_component": "v2_webhook_signing_and_delivery",
  "next_questions": [
    "Did the 502 errors start exactly at 10:42 UTC and are they affecting all EU destinations or only specific webhook endpoints?",
    "Could you provide a small, anonymized webhook delivery log excerpt (timestamp, endpoint URL, HTTP status, and whether signature verification passed) for the affected requests?"
  ]
}


In [25]:
r2 = client.responses.create(
    model="gpt-5-nano",
    previous_response_id=r1.id,
    input="Write a customer-facing email that acknowledges impact, says what we’re checking, and asks exactly two specific questions.",
    instructions="Write plain English email text and do not output JSON.",
)

print(r2.output_text)

Subject: Update on EU webhook impact after v2 signing

Hello,

We understand that since 10:42 UTC you’ve been seeing webhook retries and multiple 502 errors, and that this started after enabling v2 signing. We’re sorry for the disruption this is causing your EU customers.

What we’re checking
- Our team is investigating the webhook delivery path related to the v2 signing update, including signature verification and retry behavior, as well as any routing or endpoint issues affecting EU destinations.
- We’re reviewing recent EU-bound deliveries to identify patterns (such as specific endpoints or time windows) and to determine the scope of impact.

Two quick questions to help us triage
1) Are the 502 errors affecting all EU destinations, or only specific webhook endpoints?
2) Could you share a small anonymized webhook delivery sample that includes: timestamp, endpoint URL (redacted if needed), HTTP status, and whether the v2 signature verification passed?

Thank you for your patience. We’

What this demonstrates in practice:

1. The developer policy stays in effect across both calls (because it’s part of the thread).

2. The instructions cleanly switch “mode” (JSON → email) because instructions are per-call and aren’t carried forward when you use previous_response_id.

## Scenario 2: One DB assistant, multiple “surfaces” (Slack answer → Jira incident update → internal runbook)

Why you need both:
Same knowledge + rules, but each surface needs a different output contract (short Slack reply, structured Jira update, detailed runbook). You don’t want those formatting rules to permanently pollute the thread as more developer messages; you want them to be ephemeral and swapped per action.

In [26]:
r1 = client.responses.create(
    model="gpt-5-nano",
    input=[
        {"role": "developer", "content": "You are ACME Incident Assistant; do not guess unknown facts; if unsure say what you need; keep recommendations actionable; do not expose internal-only details."},
        {"role": "user", "content": "We’re seeing intermittent payment failures in EU; gateway 502 spike started 08:12 UTC; failover reduced it but not fully."},
    ],
    instructions="Output as a Slack message with at most 6 lines and include one short checklist.",
)

print(r1.output_text)

EU payments: intermittent failures; gateway 502 spike started 08:12 UTC; failover reduced but not fully.
Status: investigating; impact limited to EU region; some payment attempts still fail.
Likely areas to inspect (no guesses): gateway health, failover routing, processor status, and downstream services; monitor 502 patterns.
Needed from you: latest gateway/processor logs since 08:12 UTC; confirm affected endpoints (all EU or subregions); any recent config changes.
Next steps: validate mitigations, monitor KPIs, and plan stakeholder updates.
Checklist: gather logs; confirm scope; publish ETA.


In [27]:
r2 = client.responses.create(
    model="gpt-5-nano",
    previous_response_id=r1.id,
    input="Convert this into a Jira incident update.",
    instructions="Return only JSON with keys summary, customer_impact, current_status, next_actions.",
)

print(r2.output_text)

{
  "summary": "EU Payments: Intermittent failures due to gateway 502 spike; failover partially mitigated since 08:12 UTC",
  "customer_impact": "- Intermittent payment failures for EU region; some transactions failing despite failover.\n- No confirmed global outages.",
  "current_status": "Status: Investigating. Gateway 502 spike observed; started 08:12 UTC. Failover reduced load but did not fully resolve. Impact localized to EU region; diagnostics ongoing to identify root cause and verify affected endpoints and downstream processors.",
  "next_actions": "- Collect latest gateway and processor logs since 08:12 UTC for analysis.\n- Confirm affected scope: all EU endpoints or subregions; verify any downstream processors involved.\n- Review recent configuration changes or deployments that could relate to 502 errors.\n- Validate failover routing and resiliency; identify any partial failures in routing.\n- Monitor KPIs (502 rate, latency, throughput, failover success) and update dashboards

In [28]:
r3 = client.responses.create(
    model="gpt-5-nano",
    previous_response_id=r2.id,
    input="Now write the on-call runbook section for investigating this failure pattern.",
    instructions="Write Markdown with headings and include example commands and what signals to look for.",
)

print(r3.output_text)

# On-Call Runbook: Investigating EU Intermittent Payment Failures (Gateway 502 Spike)

Purpose
- Provide a disciplined, repeatable process for investigating intermittent payment failures in the EU region caused by gateway 502 errors.
- Help the on-call engineer quickly confirm scope, gather evidence, test hypotheses, implement mitigations, and communicate status.

Scope
- Incident pattern: 502 responses from the payment gateway with partial failover, affecting EU region(s) only.
- Time window to review: starting at the reported spike time (08:12 UTC) and ongoing.

Severity and Escalation
- Treat as Sev-1 if payment processing is broadly unusable for EU customers or exceeds your service-level targets.
- If uncertainty about impact, start at Sev-1 criteria and adjust as data comes in.
- Escalate to the on-call manager if you cannot determine a path to remediation within 30–60 minutes or if core upstream processors are unavailable.

Triage Flow (step-by-step)
1) Verify alert and scope
   

![Illustration](https://github.com/shivam13juna/language_model_api_v2/blob/main/llm_multi_modality/instruction_vs_developer.png)


# How to handle images

In [35]:

import base64

# Read and encode the image file
image_path = '/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/llm_conversations/ss.jpeg'
with open(image_path, 'rb') as img_file:
    image_data = base64.standard_b64encode(img_file.read()).decode('utf-8')

# Use chat.completions.create with vision
vision_response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {
            "role": "system",
            "content": "You are an art critic who provides gentle feedback for children's illustrations."
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Analyze this magical  illustration."
                },
                {
                    "type": "image_url",
                    #"image_url": {
                    #    "url": f"data:image/jpeg;base64,{image_data}"
                    #}
                    "image_url": {
                        "url": "https://www.animesenpai.net/wp-content/uploads/2023/12/sef-min.png.webp",
                        "detail": "low" # other options are "medium" and "high", "original"
                    }
                }
            ]
        }
    ],
    #temperature=0.5
)

pretty_print("\n=== Vision Analysis ===")
pretty_print(vision_response.choices[0].message.content)
# print token
print("\nPrompt tokens:", vision_response.usage.prompt_tokens)
print("Completion tokens:", vision_response.usage.completion_tokens)
print("Total tokens:", vision_response.usage.total_tokens)

 === Vision Analysis ===
What a striking, magical moment you’ve captured. The figure feels powerful and
otherworldly, and the misty backdrop really sells a dreamlike night scene.  What
works well - Silhouette and pose: The tall, broad-shouldered form with
outstretched limbs reads as a guardian or sentinel. The dynamic stance gives a
sense of motion and presence. - Contrast and depth: The pale blue-gray body
against the dark, smoky background creates a strong read at a distance. The fog
adds depth and mystery. - Lighting: A cool, directional light highlights the
chest and arms, giving the form volume and emphasizing the creature’s
musculature in a dramatic way. - Color restraint: Sticking to a cool,
monochromatic palette helps the illustration feel cohesive and otherworldly.
Areas to consider for a children's-friendly version - Expressiveness: The face
and eyes feel intense. For a gentler mood, soften the expression—larger, rounder
eyes; a small, calm smile; or a friendlier gaze. This m

In [36]:

import base64

# Read and encode the image file
image_path = '/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/llm_conversations/ss.jpeg'
with open(image_path, 'rb') as img_file:
    image_data = base64.standard_b64encode(img_file.read()).decode('utf-8')

# Use chat.completions.create with vision
vision_response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {
            "role": "system",
            "content": "You are an art critic who provides gentle feedback for children's illustrations."
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Analyze this magical  illustration."
                },
                {
                    "type": "image_url",
                    #"image_url": {
                    #    "url": f"data:image/jpeg;base64,{image_data}"
                    #}
                    "image_url": {
                        "url": "https://www.animesenpai.net/wp-content/uploads/2023/12/sef-min.png.webp",
                        "detail": "high" # other options are "medium" and "high", "original"
                    }
                }
            ]
        }
    ],
    #temperature=0.5
)

pretty_print("\n=== Vision Analysis ===")
pretty_print(vision_response.choices[0].message.content)
# print token
print("\nPrompt tokens:", vision_response.usage.prompt_tokens)
print("Completion tokens:", vision_response.usage.completion_tokens)
print("Total tokens:", vision_response.usage.total_tokens)

 === Vision Analysis ===
What a striking, magical moment this image captures. It feels cinematic and full
of story, even at a single glance.  What works well - Mood and atmosphere: The
cool, night-time palette and the smoky fog create a sense of mystery and awe
that feels like a fairy-tantasy chapter opening. - Silhouette and design: The
creature’s bold, angular silhouette reads clearly from a distance. The puffed
chest, elongated arms, and the curved tail lead the eye in a dynamic arc, giving
the figure a sense of power and otherworldliness. - Texture and detailing: The
rib-like chest pattern and the musculature read as sculpted, almost bone-white,
which heightens the magical, otherworldly vibe. The wrist/arm bands and the
wispy, wing-like extensions add variety in texture. - Depth and composition: The
foreground smoke plus the blurred background lights create depth and a sense of
scale—the figure feels monumental and a bit magical, not just a flat silhouette.
Suggestions for making i

In [37]:
response_vision = client.responses.create(
    model="gpt-5-nano",
    instructions="You are an art critic who provides gentle feedback for children's illustrations.",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": "Analyze this magical illustration."
                },
                {
                    "type": "input_image",
                    "image_url": "https://www.animesenpai.net/wp-content/uploads/2023/12/sef-min.png.webp",
                    "detail": "low"
                }
            ]
        }
    ]
)

pretty_print("\n=== Vision Analysis (Responses API) ===")
pretty_print(response_vision.output_text)

print("\nInput tokens:", response_vision.usage.input_tokens)
print("Output tokens:", response_vision.usage.output_tokens)
print("Total tokens:", response_vision.usage.total_tokens)

 === Vision Analysis (Responses API) ===
What a striking, atmospheric image. It has a strong sense of magic and awe that
would draw a reader into a forest or moonlit world.  What works well - Bold
silhouette and pose: The broad shoulders, extended arms, and long tail create a
dynamic, heroic silhouette that reads clearly even from a distance. - Contrast
and mood: The cool, bluish tones against a dark background with soft mist gives
a mysterious, otherworldly feel. The fog helps push the figure forward. -
Texture and design: The rib-like chest and the jagged, armor-like shapes give
the character a unique, fantasy-creature presence. The subtle highlights help
the muscles and form pop. - Story potential: The image hints at a guardian or
ancient being watching over a magical place—ripe for a story about courage,
discovery, or friendship.  Gentle tweaks to consider (kid-friendly and story-
enhancing) - soften the facial expression: a gentler look or a hint of a smile
can make the character 

# Text to Speech

In [40]:
from pathlib import Path

speech_file_path = Path("tutor_voice.mp3")

with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="nova",  # e.g. alloy, verse, aria (varies)
    input="Hello students! Today we will explore AI that can see, listen, and speak."
) as response:
    response.stream_to_file(speech_file_path)

print("Saved speech to:", speech_file_path)


Saved speech to: tutor_voice.mp3


# Speech to Text

In [43]:
from pathlib import Path

audio_file_path = Path("/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/other_modalities/test_audio.wav")

with open(audio_file_path, "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model="gpt-4o-transcribe",
        file=audio_file,
    )

print("Transcribed text:", transcription.text)


if transcription.usage:
    print("Input tokens:", getattr(transcription.usage, "input_tokens", None))
    print("Output tokens:", getattr(transcription.usage, "output_tokens", None))
    print("Total tokens:", getattr(transcription.usage, "total_tokens", None))

    details = getattr(transcription.usage, "input_token_details", None)
    if details:
        print("Audio tokens:", getattr(details, "audio_tokens", None))
        print("Text tokens:", getattr(details, "text_tokens", None))

Transcribed text: Hello testing one two three
Input tokens: 34
Output tokens: 7
Total tokens: 41
Audio tokens: 34
Text tokens: 0


In [44]:
from pathlib import Path

audio_file_path = Path("/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/other_modalities/test_audio.wav")

guiding_text = "This is a simple English test recording. Please transcribe it accurately."

with open(audio_file_path, "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model="gpt-4o-transcribe",
        file=audio_file,
        prompt=guiding_text,
    )

print("Transcribed text:", transcription.text)

if transcription.usage:
    print("Input tokens:", getattr(transcription.usage, "input_tokens", None))
    print("Output tokens:", getattr(transcription.usage, "output_tokens", None))
    print("Total tokens:", getattr(transcription.usage, "total_tokens", None))

    details = getattr(transcription.usage, "input_token_details", None)
    if details:
        print("Audio tokens:", getattr(details, "audio_tokens", None))
        print("Text tokens:", getattr(details, "text_tokens", None))

Transcribed text: Hello, testing one two three
Input tokens: 47
Output tokens: 8
Total tokens: 55
Audio tokens: 34
Text tokens: 13


# Some Bonus Content

## Image Generation

```python
from openai import OpenAI
import os
import requests

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

prompt = "A friendly robot tutor teaching Python in a bright classroom, cartoon style"

result = client.images.generate(
    model="gpt-image-1",
    prompt=prompt,
    size="512x512",
    quality="standard",
    n=1
)

image_url = result.data[0].url
print("Image URL:", image_url)

# Optional: Save locally
img_data = requests.get(image_url).content
with open("robot_tutor.png", "wb") as f:
    f.write(img_data)



## Text to Speech

```python

from pathlib import Path

speech_file_path = Path("tutor_voice.mp3")

with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="alloy",  # e.g. alloy, verse, aria (varies)
    input="Hello students! Today we will explore AI that can see, listen, and speak."
) as response:
    response.stream_to_file(speech_file_path)

print("Saved speech to:", speech_file_path)



## Speech to Text

```python

from pathlib import Path

audio_file_path = Path("student_question.mp3")

with client.audio.transcriptions.create(
    model="gpt-4o-transcribe",
    file=open(audio_file_path, "rb")
) as transcription:
    print("Transcribed text:", transcription.text)



### Speech to Text with response

```python

# Step 1: Transcribe
with client.audio.transcriptions.create(
    model="gpt-4o-transcribe",
    file=open(audio_file_path, "rb")
) as transcription:
    user_text = transcription.text

# Step 2: Feed into Chat API
messages = [{"role": "system", "content": "You are a helpful multimodal tutor."},
            {"role": "user", "content": user_text}]

resp = client.chat.completions.create(model="gpt-4o-mini", messages=messages)
print("Assistant:", resp.choices[0].message.content)



## Visual Reasoning with GROQ

```python

from groq import Groq
import os

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3b/Example.png/320px-Example.png"

messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe this image in detail."},
            {"type": "image_url", "image_url": {"url": image_url}}
        ]
    }
]

resp = groq_client.chat.completions.create(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    messages=messages
)

print("Image analysis:", resp.choices[0].message.content)



## PlayHT

```python
import requests

PLAYHT_API_KEY = os.getenv("PLAYHT_API_KEY")
PLAYHT_USER_ID = os.getenv("PLAYHT_USER_ID")

url = "https://api.play.ht/api/v2/tts"
headers = {
    "Authorization": f"Bearer {PLAYHT_API_KEY}",
    "X-User-Id": PLAYHT_USER_ID,
    "Content-Type": "application/json"
}

payload = {
    "voice": "en_us_male_1",
    "content": ["Hello! I can speak with a PlayHT voice."],
    "format": "mp3"
}

response = requests.post(url, headers=headers, json=payload)
print(response.json())  # Contains URL to generated audio